# 📊 Data Visualization for Machine Learning

**Part of the Python ML Study Guide - Phase 2: Core Fundamentals**

---

## 🎯 Learning Objectives

By the end of this notebook, you will:

1. Create **publication-quality plots** with Matplotlib
2. Build **statistical visualizations** with Seaborn
3. Visualize **distributions** and detect outliers
4. Explore **relationships** between features
5. Create **ML-specific plots** (learning curves, confusion matrices)
6. Understand **when to use which plot type**

---

## 📚 Table of Contents

1. [Introduction to Visualization](#1-introduction)
2. [Matplotlib Fundamentals](#2-matplotlib-fundamentals)
3. [Distribution Plots](#3-distribution-plots)
4. [Relationship Plots](#4-relationship-plots)
5. [Categorical Plots](#5-categorical-plots)
6. [Heatmaps and Correlation](#6-heatmaps-and-correlation)
7. [ML-Specific Visualizations](#7-ml-specific-visualizations)
8. [Practice Exercises](#8-practice-exercises)

---

## 1. Introduction to Visualization

### Why Visualization Matters for ML

| Purpose | What It Reveals | When to Use |
|---------|-----------------|-------------|
| **EDA** | Data patterns, anomalies | Before modeling |
| **Feature Understanding** | Distributions, relationships | Feature selection |
| **Model Debugging** | Predictions vs. actuals | During development |
| **Communication** | Results to stakeholders | After modeling |

### The Visualization Stack

```
High-level (easy)    ┌─────────────┐
                     │   Seaborn   │  Statistical plots
                     ├─────────────┤
                     │ Matplotlib  │  Full control
Low-level (flexible) └─────────────┘
```

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure defaults
plt.style.use('seaborn-v0_8-whitegrid')  # Clean style
plt.rcParams['figure.figsize'] = [10, 6]  # Default size
plt.rcParams['figure.dpi'] = 100  # Resolution
sns.set_palette('husl')  # Color palette

# For reproducibility
np.random.seed(42)

print(f'Matplotlib version: {plt.matplotlib.__version__}')
print(f'Seaborn version: {sns.__version__}')

In [ ]:
# Create sample dataset for visualization
n_samples = 200

df = pd.DataFrame({
    'age': np.random.normal(35, 10, n_samples).clip(18, 70).astype(int),
    'income': np.random.exponential(50000, n_samples).clip(20000, 200000),
    'education_years': np.random.choice([12, 14, 16, 18, 20], n_samples, p=[0.3, 0.25, 0.25, 0.15, 0.05]),
    'department': np.random.choice(['Sales', 'Engineering', 'Marketing', 'HR'], n_samples),
    'satisfaction': np.random.uniform(1, 10, n_samples).round(1),
    'tenure_years': np.random.exponential(3, n_samples).clip(0, 20).round(1)
})

# Add correlated feature
df['performance'] = (0.3 * df['satisfaction'] + 
                     0.2 * df['education_years']/4 + 
                     np.random.normal(0, 1, n_samples)).clip(1, 10).round(1)

print('Sample Dataset:')
print(df.head(10))
print(f'\nShape: {df.shape}')

---

## 2. Matplotlib Fundamentals

### The Figure and Axes Model

```
┌─────────────────────────────────────────┐
│  Figure (container)                     │
│  ┌─────────────────┐ ┌─────────────────┐│
│  │ Axes 1          │ │ Axes 2          ││
│  │ (subplot)       │ │ (subplot)       ││
│  │                 │ │                 ││
│  └─────────────────┘ └─────────────────┘│
└─────────────────────────────────────────┘
```

**Key concept**: Always use the object-oriented interface (`fig, ax = plt.subplots()`)

In [ ]:
# Basic line plot
fig, ax = plt.subplots(figsize=(10, 5))

x = np.linspace(0, 10, 100)
ax.plot(x, np.sin(x), label='sin(x)', linewidth=2)
ax.plot(x, np.cos(x), label='cos(x)', linewidth=2, linestyle='--')

ax.set_xlabel('X axis', fontsize=12)
ax.set_ylabel('Y axis', fontsize=12)
ax.set_title('Basic Line Plot', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Multiple subplots
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Plot 1: Line
axes[0, 0].plot(x, np.sin(x), color='blue')
axes[0, 0].set_title('Line Plot')

# Plot 2: Scatter
axes[0, 1].scatter(df['age'], df['income'], alpha=0.5, c='green')
axes[0, 1].set_title('Scatter Plot')
axes[0, 1].set_xlabel('Age')
axes[0, 1].set_ylabel('Income')

# Plot 3: Bar
dept_counts = df['department'].value_counts()
axes[1, 0].bar(dept_counts.index, dept_counts.values, color='orange')
axes[1, 0].set_title('Bar Plot')
axes[1, 0].tick_params(axis='x', rotation=45)

# Plot 4: Histogram
axes[1, 1].hist(df['satisfaction'], bins=20, color='purple', edgecolor='black')
axes[1, 1].set_title('Histogram')
axes[1, 1].set_xlabel('Satisfaction Score')

plt.tight_layout()
plt.show()

In [ ]:
# Customizing plot appearance
fig, ax = plt.subplots(figsize=(10, 6))

# Plot with extensive customization
scatter = ax.scatter(
    df['education_years'], 
    df['income'],
    c=df['satisfaction'],  # Color by satisfaction
    s=df['tenure_years'] * 10,  # Size by tenure
    cmap='viridis',
    alpha=0.6,
    edgecolors='white',
    linewidth=0.5
)

# Add colorbar
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Satisfaction Score', fontsize=11)

# Labels and title
ax.set_xlabel('Education (Years)', fontsize=12)
ax.set_ylabel('Income ($)', fontsize=12)
ax.set_title('Income vs Education\n(Color: Satisfaction, Size: Tenure)', 
             fontsize=14, fontweight='bold')

# Format y-axis as currency
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

---

## 3. Distribution Plots

Understanding data distributions is crucial for:
- Choosing appropriate ML models
- Deciding on normalization/standardization
- Detecting outliers
- Feature engineering decisions

| Plot Type | Use Case | Key Insight |
|-----------|----------|-------------|
| Histogram | Overall distribution | Shape, skewness |
| KDE | Smooth distribution | Probability density |
| Box Plot | Quartiles + outliers | IQR, extreme values |
| Violin | Distribution + density | Full shape comparison |

In [ ]:
# Histogram with KDE overlay
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Income distribution (right-skewed)
sns.histplot(df['income'], kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Income Distribution\n(Right-skewed)', fontweight='bold')
axes[0].set_xlabel('Income ($)')

# Age distribution (normal-ish)
sns.histplot(df['age'], kde=True, ax=axes[1], color='coral')
axes[1].set_title('Age Distribution\n(Approximately Normal)', fontweight='bold')
axes[1].set_xlabel('Age')

# Satisfaction distribution
sns.histplot(df['satisfaction'], kde=True, ax=axes[2], color='seagreen')
axes[2].set_title('Satisfaction Distribution\n(Uniform)', fontweight='bold')
axes[2].set_xlabel('Satisfaction Score')

plt.tight_layout()
plt.show()

In [ ]:
# Box plots - great for detecting outliers
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Single box plot
sns.boxplot(y=df['income'], ax=axes[0], color='lightblue')
axes[0].set_title('Income Box Plot\n(Outliers shown as diamonds)', fontweight='bold')
axes[0].set_ylabel('Income ($)')

# Box plot by category
sns.boxplot(x='department', y='income', data=df, ax=axes[1], palette='Set2')
axes[1].set_title('Income by Department', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Violin plots - combines box plot + KDE
fig, ax = plt.subplots(figsize=(12, 6))

sns.violinplot(
    x='department', 
    y='satisfaction', 
    data=df, 
    palette='muted',
    inner='box'  # Shows box plot inside
)

ax.set_title('Satisfaction Distribution by Department\n(Violin = density, inner box = quartiles)', 
             fontweight='bold', fontsize=13)
ax.set_xlabel('Department')
ax.set_ylabel('Satisfaction Score')

plt.tight_layout()
plt.show()

In [ ]:
# Comparing distributions: Multiple KDEs
fig, ax = plt.subplots(figsize=(10, 6))

for dept in df['department'].unique():
    subset = df[df['department'] == dept]
    sns.kdeplot(subset['performance'], label=dept, ax=ax, linewidth=2)

ax.set_title('Performance Distribution by Department', fontweight='bold', fontsize=13)
ax.set_xlabel('Performance Score')
ax.set_ylabel('Density')
ax.legend(title='Department')

plt.tight_layout()
plt.show()

In [ ]:
# Identifying skewness - important for ML preprocessing
from scipy import stats

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Original (skewed)
skewness = stats.skew(df['income'])
sns.histplot(df['income'], kde=True, ax=axes[0], color='salmon')
axes[0].set_title(f'Original Income\nSkewness: {skewness:.2f}', fontweight='bold')
axes[0].axvline(df['income'].mean(), color='red', linestyle='--', label='Mean')
axes[0].axvline(df['income'].median(), color='blue', linestyle='--', label='Median')
axes[0].legend()

# Log-transformed (more normal)
log_income = np.log1p(df['income'])
skewness_log = stats.skew(log_income)
sns.histplot(log_income, kde=True, ax=axes[1], color='lightgreen')
axes[1].set_title(f'Log-Transformed Income\nSkewness: {skewness_log:.2f}', fontweight='bold')
axes[1].set_xlabel('log(Income + 1)')

plt.tight_layout()
plt.show()

print('💡 ML Tip: Many models (linear regression, neural networks) work better with normally distributed features.')
print('   Use log transform for right-skewed data, or consider Box-Cox transformation.')

---

## 4. Relationship Plots

Understanding relationships between features helps with:
- Feature selection (highly correlated features can be redundant)
- Model choice (linear vs. non-linear)
- Identifying potential feature interactions

| Plot | Variables | Insight |
|------|-----------|--------|
| Scatter | 2 numeric | Linear/non-linear relationship |
| Pair Plot | Multiple numeric | All pairwise relationships |
| Joint Plot | 2 numeric | Relationship + marginal distributions |
| Reg Plot | 2 numeric | Relationship + regression line |

In [ ]:
# Scatter plot with trend line
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter with regression line
sns.regplot(x='satisfaction', y='performance', data=df, ax=axes[0],
            scatter_kws={'alpha': 0.5}, line_kws={'color': 'red'})
axes[0].set_title('Satisfaction vs Performance\n(with regression line)', fontweight='bold')

# Scatter colored by category
for dept in df['department'].unique():
    subset = df[df['department'] == dept]
    axes[1].scatter(subset['tenure_years'], subset['income'], 
                    label=dept, alpha=0.6, s=50)

axes[1].set_title('Tenure vs Income by Department', fontweight='bold')
axes[1].set_xlabel('Tenure (Years)')
axes[1].set_ylabel('Income ($)')
axes[1].legend(title='Department')

plt.tight_layout()
plt.show()

In [ ]:
# Pair plot - visualize all pairwise relationships
# Select numeric columns for pairplot
numeric_cols = ['age', 'income', 'satisfaction', 'performance']

# Create pair plot (can be slow for large datasets)
g = sns.pairplot(
    df[numeric_cols + ['department']], 
    hue='department',
    diag_kind='kde',
    plot_kws={'alpha': 0.6},
    height=2.5
)

g.figure.suptitle('Pair Plot: All Numeric Relationships by Department', 
                 y=1.02, fontweight='bold', fontsize=14)
plt.show()

print('💡 ML Tip: Pair plots are great for EDA but can be slow with many features.')
print('   Use correlation heatmaps for quick overview of many features.')

In [ ]:
# Joint plot - detailed view of two variables
g = sns.jointplot(
    x='education_years', 
    y='income', 
    data=df,
    kind='scatter',  # Options: 'scatter', 'kde', 'hist', 'hex', 'reg'
    height=8,
    marginal_kws={'fill': True}
)

g.figure.suptitle('Education vs Income\n(with marginal distributions)', 
                 y=1.02, fontweight='bold')
plt.show()

In [ ]:
# Hexbin plot for large datasets (avoids overplotting)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Regular scatter (can have overplotting)
axes[0].scatter(df['age'], df['income'], alpha=0.3)
axes[0].set_title('Scatter Plot\n(Overplotting issue)', fontweight='bold')
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Income')

# Hexbin (shows density)
hb = axes[1].hexbin(df['age'], df['income'], gridsize=15, cmap='YlOrRd')
axes[1].set_title('Hexbin Plot\n(Shows density)', fontweight='bold')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Income')
plt.colorbar(hb, ax=axes[1], label='Count')

plt.tight_layout()
plt.show()

print('💡 ML Tip: Use hexbin or 2D KDE for datasets with 10,000+ points to avoid overplotting.')

---

## 5. Categorical Plots

Categorical plots help understand:
- Distribution of numeric variables across categories
- Class imbalance in classification targets
- How to encode categorical features

| Plot | Purpose | When to Use |
|------|---------|-------------|
| Bar | Count/mean by category | Compare categories |
| Count | Category frequencies | Check class balance |
| Box/Strip | Distribution by category | Compare spreads |
| Swarm | Individual points by category | Small datasets |

In [ ]:
# Bar plots for categorical comparisons
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Count plot - check class balance
sns.countplot(x='department', data=df, ax=axes[0], palette='Set2')
axes[0].set_title('Department Distribution\n(Check for class imbalance)', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Bar plot - mean by category
sns.barplot(x='department', y='performance', data=df, ax=axes[1], 
            palette='Set2', estimator='mean', errorbar='sd')
axes[1].set_title('Mean Performance by Department\n(with std error bars)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

# Horizontal bar (good for many categories)
dept_income = df.groupby('department')['income'].mean().sort_values()
axes[2].barh(dept_income.index, dept_income.values, color='steelblue')
axes[2].set_title('Mean Income by Department\n(sorted)', fontweight='bold')
axes[2].set_xlabel('Income ($)')

plt.tight_layout()
plt.show()

In [ ]:
# Box and strip plots by category
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Box plot by category
sns.boxplot(x='department', y='income', data=df, ax=axes[0], palette='Set3')
axes[0].set_title('Income Distribution by Department\n(Box Plot)', fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)

# Strip plot (shows individual points)
sns.stripplot(x='department', y='income', data=df, ax=axes[1], 
              palette='Set3', alpha=0.6, jitter=True)
axes[1].set_title('Income by Department\n(Strip Plot - individual points)', fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Combined box + swarm plot (best of both worlds)
plt.figure(figsize=(10, 6))

# Box plot for distribution summary
sns.boxplot(x='department', y='satisfaction', data=df, 
            palette='pastel', width=0.5)

# Overlay swarm plot for individual points
sns.swarmplot(x='department', y='satisfaction', data=df, 
              color='black', alpha=0.5, size=3)

plt.title('Satisfaction by Department\n(Box + Swarm Plot)', fontweight='bold', fontsize=14)
plt.xlabel('Department', fontsize=12)
plt.ylabel('Satisfaction Score', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print('💡 ML Tip: Combined plots show both summary statistics AND individual observations.')

In [ ]:
# Checking class imbalance (critical for classification)
# Create a binary target variable for demonstration
df['high_performer'] = (df['performance'] >= df['performance'].median()).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
class_counts = df['high_performer'].value_counts()
axes[0].bar(['Low Performer (0)', 'High Performer (1)'], 
            class_counts.values, color=['salmon', 'lightgreen'])
axes[0].set_title('Target Variable Distribution\n(Check for imbalance)', fontweight='bold')
axes[0].set_ylabel('Count')

# Add percentage annotations
for i, v in enumerate(class_counts.values):
    pct = v / len(df) * 100
    axes[0].text(i, v + 2, f'{pct:.1f}%', ha='center', fontweight='bold')

# Pie chart for proportions
axes[1].pie(class_counts.values, labels=['Low', 'High'], autopct='%1.1f%%',
            colors=['salmon', 'lightgreen'], startangle=90, explode=[0.02, 0.02])
axes[1].set_title('Class Balance', fontweight='bold')

plt.tight_layout()
plt.show()

# Imbalance ratio
ratio = class_counts.max() / class_counts.min()
print(f'📊 Class imbalance ratio: {ratio:.2f}:1')
if ratio > 3:
    print('⚠️  Warning: Significant imbalance! Consider SMOTE, undersampling, or class weights.')
else:
    print('✅ Classes are reasonably balanced.')

---

## 6. Heatmaps and Correlation Analysis

Correlation analysis is crucial for ML:
- **Feature Selection**: Identify highly correlated features (redundant)
- **Multicollinearity**: High correlation between features can hurt linear models
- **Target Correlation**: Find features most predictive of target

| Correlation | Strength | Interpretation |
|-------------|----------|----------------|
| 0.0 - 0.3 | Weak | Features are independent |
| 0.3 - 0.7 | Moderate | Some relationship |
| 0.7 - 1.0 | Strong | Features share information |

In [ ]:
# Compute correlation matrix
numeric_df = df.select_dtypes(include=[np.number])
correlation_matrix = numeric_df.corr()

print('Correlation Matrix:')
print(correlation_matrix.round(2))
print()

# Find highly correlated pairs
print('Highly correlated feature pairs (|r| > 0.5):')
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr = correlation_matrix.iloc[i, j]
        if abs(corr) > 0.5:
            print(f'  {correlation_matrix.columns[i]} ↔ {correlation_matrix.columns[j]}: {corr:.3f}')

In [ ]:
# Basic correlation heatmap
plt.figure(figsize=(10, 8))

sns.heatmap(
    correlation_matrix,
    annot=True,           # Show correlation values
    fmt='.2f',            # Format to 2 decimal places
    cmap='RdBu_r',        # Red-Blue diverging colormap
    center=0,             # Center colormap at 0
    square=True,          # Square cells
    linewidths=0.5,       # Cell border width
    vmin=-1, vmax=1       # Full correlation range
)

plt.title('Correlation Heatmap\n(Red=Positive, Blue=Negative)', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Lower triangle heatmap (cleaner for many features)
# Create mask for upper triangle
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

plt.figure(figsize=(10, 8))

sns.heatmap(
    correlation_matrix,
    mask=mask,            # Hide upper triangle
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={'shrink': 0.8, 'label': 'Correlation'}
)

plt.title('Correlation Heatmap (Lower Triangle)\nCleaner view for many features', 
          fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

print('💡 ML Tip: Use lower triangle to avoid redundancy and clutter.')

In [ ]:
# Feature correlation with target variable
target = 'high_performer'

# Correlations with target (sorted)
target_corr = correlation_matrix[target].drop(target).sort_values(key=abs, ascending=False)

plt.figure(figsize=(10, 6))

# Color bars by positive/negative correlation
colors = ['green' if c > 0 else 'red' for c in target_corr.values]

bars = plt.barh(target_corr.index, target_corr.values, color=colors, edgecolor='black')

# Add value labels
for bar, val in zip(bars, target_corr.values):
    x_pos = val + 0.02 if val >= 0 else val - 0.08
    plt.text(x_pos, bar.get_y() + bar.get_height()/2, f'{val:.3f}', va='center')

plt.axvline(x=0, color='black', linewidth=0.5)
plt.xlabel('Correlation with Target', fontsize=12)
plt.title(f'Feature Correlations with Target ({target})\nGreen=Positive, Red=Negative', 
          fontweight='bold', fontsize=14)
plt.xlim(-1, 1)
plt.tight_layout()
plt.show()

print('\n💡 ML Tip: Features with |correlation| < 0.1 may not be predictive.')
print('   But correlation only captures LINEAR relationships!')

In [ ]:
# Cluster heatmap - groups similar features together
g = sns.clustermap(
    correlation_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    figsize=(10, 10),
    linewidths=0.5,
    dendrogram_ratio=(0.1, 0.1)
)

g.figure.suptitle('Clustered Correlation Heatmap\n(Groups similar features)', 
                 y=1.02, fontweight='bold', fontsize=14)
plt.show()

print('💡 ML Tip: Clustered heatmaps reveal groups of correlated features.')
print('   Consider keeping only one feature from each cluster to reduce dimensionality.')

---

## 7. ML-Specific Visualizations

These plots are specifically designed for ML workflows:

| Plot | Stage | Purpose |
|------|-------|--------|
| Missing Value Heatmap | Data Cleaning | Identify missing patterns |
| Feature Importance | Post-Training | Understand model decisions |
| Confusion Matrix | Evaluation | Classification performance |
| Learning Curves | Evaluation | Diagnose overfitting/underfitting |
| ROC/PR Curves | Evaluation | Binary classification metrics |

In [ ]:
# Missing value visualization
# Create some missing values for demonstration
df_missing = df.copy()
np.random.seed(42)
missing_mask = np.random.random(df_missing.shape) < 0.1  # 10% missing

# Only apply to numeric columns
for i, col in enumerate(df_missing.columns):
    if df_missing[col].dtype in ['int64', 'float64']:
        df_missing.loc[missing_mask[:, i], col] = np.nan

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Missing value heatmap
sns.heatmap(df_missing.isnull(), cbar=True, yticklabels=False, ax=axes[0],
            cmap='viridis')
axes[0].set_title('Missing Value Pattern\n(Yellow = Missing)', fontweight='bold')
axes[0].set_xlabel('Features')

# Missing percentage bar chart
missing_pct = (df_missing.isnull().sum() / len(df_missing) * 100).sort_values(ascending=True)
missing_pct = missing_pct[missing_pct > 0]  # Only show columns with missing

colors = ['green' if p < 5 else 'orange' if p < 20 else 'red' for p in missing_pct.values]
axes[1].barh(missing_pct.index, missing_pct.values, color=colors)
axes[1].axvline(x=5, color='green', linestyle='--', label='5% threshold')
axes[1].axvline(x=20, color='red', linestyle='--', label='20% threshold')
axes[1].set_title('Missing Value Percentage by Feature', fontweight='bold')
axes[1].set_xlabel('Missing %')
axes[1].legend()

plt.tight_layout()
plt.show()

print('💡 ML Tips for missing values:')
print('   - <5% missing: Simple imputation usually fine')
print('   - 5-20% missing: Consider more sophisticated imputation')
print('   - >20% missing: Consider dropping feature or using models that handle missing')

In [ ]:
# Feature importance visualization (simulated)
# In real ML, this comes from trained models
features = ['income', 'satisfaction', 'tenure_years', 'education_years', 'age', 'performance']
importance = np.array([0.35, 0.25, 0.15, 0.12, 0.08, 0.05])

# Sort by importance
sorted_idx = np.argsort(importance)
features_sorted = [features[i] for i in sorted_idx]
importance_sorted = importance[sorted_idx]

plt.figure(figsize=(10, 6))

# Horizontal bar chart
bars = plt.barh(features_sorted, importance_sorted, color='steelblue', edgecolor='black')

# Color top features differently
bars[-1].set_color('darkgreen')
bars[-2].set_color('forestgreen')

# Add value labels
for bar, val in zip(bars, importance_sorted):
    plt.text(val + 0.01, bar.get_y() + bar.get_height()/2, 
             f'{val:.2f}', va='center', fontweight='bold')

plt.xlabel('Feature Importance', fontsize=12)
plt.title('Feature Importance\n(from Random Forest)', fontweight='bold', fontsize=14)
plt.xlim(0, max(importance) * 1.2)
plt.tight_layout()
plt.show()

print('💡 ML Tip: Feature importance helps with:')
print('   - Model interpretability')
print('   - Feature selection (drop low-importance features)')
print('   - Understanding what drives predictions')

In [ ]:
# Confusion Matrix visualization
# Simulated confusion matrix for binary classification
confusion = np.array([[85, 15],   # True Negatives, False Positives
                      [10, 90]])  # False Negatives, True Positives

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(confusion, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Predicted: 0', 'Predicted: 1'],
            yticklabels=['Actual: 0', 'Actual: 1'],
            cbar=False)
axes[0].set_title('Confusion Matrix (Counts)', fontweight='bold', fontsize=12)

# Normalized (percentages)
confusion_pct = confusion / confusion.sum() * 100
sns.heatmap(confusion_pct, annot=True, fmt='.1f', cmap='Blues', ax=axes[1],
            xticklabels=['Predicted: 0', 'Predicted: 1'],
            yticklabels=['Actual: 0', 'Actual: 1'],
            cbar=False)
axes[1].set_title('Confusion Matrix (Percentages)', fontweight='bold', fontsize=12)

# Add annotations to cells
for ax in axes:
    ax.add_patch(plt.Rectangle((0, 0), 1, 1, fill=False, edgecolor='green', lw=3))
    ax.add_patch(plt.Rectangle((1, 1), 1, 1, fill=False, edgecolor='green', lw=3))

plt.tight_layout()
plt.show()

# Calculate metrics
tn, fp, fn, tp = confusion.ravel()
accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * (precision * recall) / (precision + recall)

print('📊 Classification Metrics:')
print(f'   Accuracy:  {accuracy:.2%}')
print(f'   Precision: {precision:.2%} (of predicted positives, how many are correct)')
print(f'   Recall:    {recall:.2%} (of actual positives, how many were found)')
print(f'   F1 Score:  {f1:.2%} (harmonic mean of precision and recall)')

In [ ]:
# Learning Curves - diagnose overfitting/underfitting
# Simulated learning curve data
train_sizes = np.array([0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]) * 1000

# Good fit scenario
train_scores_good = 0.95 - 0.15 * np.exp(-train_sizes/300)
val_scores_good = 0.90 - 0.25 * np.exp(-train_sizes/300)

# Overfitting scenario
train_scores_overfit = np.ones_like(train_sizes) * 0.99
val_scores_overfit = 0.70 + 0.05 * np.log(train_sizes/100)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Good fit
axes[0].plot(train_sizes, train_scores_good, 'o-', color='blue', label='Training Score')
axes[0].plot(train_sizes, val_scores_good, 'o-', color='red', label='Validation Score')
axes[0].fill_between(train_sizes, train_scores_good-0.02, train_scores_good+0.02, alpha=0.2, color='blue')
axes[0].fill_between(train_sizes, val_scores_good-0.02, val_scores_good+0.02, alpha=0.2, color='red')
axes[0].set_title('Good Fit ✅\n(Scores converge)', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Training Set Size')
axes[0].set_ylabel('Score')
axes[0].legend(loc='lower right')
axes[0].set_ylim(0.5, 1.0)
axes[0].grid(True, alpha=0.3)

# Overfitting
axes[1].plot(train_sizes, train_scores_overfit, 'o-', color='blue', label='Training Score')
axes[1].plot(train_sizes, val_scores_overfit, 'o-', color='red', label='Validation Score')
axes[1].fill_between(train_sizes, train_scores_overfit-0.01, train_scores_overfit+0.01, alpha=0.2, color='blue')
axes[1].fill_between(train_sizes, val_scores_overfit-0.03, val_scores_overfit+0.03, alpha=0.2, color='red')
axes[1].set_title('Overfitting ⚠️\n(Large gap between curves)', fontweight='bold', fontsize=12)
axes[1].set_xlabel('Training Set Size')
axes[1].set_ylabel('Score')
axes[1].legend(loc='lower right')
axes[1].set_ylim(0.5, 1.0)
axes[1].grid(True, alpha=0.3)

# Add annotation for the gap
gap = train_scores_overfit[-1] - val_scores_overfit[-1]
axes[1].annotate('', xy=(950, val_scores_overfit[-1]), xytext=(950, train_scores_overfit[-1]),
                arrowprops=dict(arrowstyle='<->', color='green', lw=2))
axes[1].text(970, 0.92, f'Gap: {gap:.2f}', fontsize=10, color='green')

plt.tight_layout()
plt.show()

print('💡 Learning Curve Interpretation:')
print('   - Curves converge → Good fit, more data may not help')
print('   - Large gap → Overfitting, try regularization or simpler model')
print('   - Both scores low → Underfitting, try more complex model')

---

## 8. Customization and Best Practices

Professional visualizations require attention to:
- **Clarity**: Labels, titles, legends
- **Accessibility**: Color choices, font sizes
- **Reproducibility**: Consistent styling
- **Export Quality**: High-resolution output

In [ ]:
# Setting global styles
# Option 1: Seaborn styles
print('Available seaborn styles:', ['darkgrid', 'whitegrid', 'dark', 'white', 'ticks'])

# Option 2: Matplotlib styles
print('Available matplotlib styles:', plt.style.available[:10], '...')

# Apply a style
plt.style.use('seaborn-v0_8-whitegrid')  # Clean scientific style

# Set seaborn context for different output sizes
sns.set_context('notebook')  # Options: 'paper', 'notebook', 'talk', 'poster'

# Set color palette
sns.set_palette('husl')  # Options: 'deep', 'muted', 'bright', 'pastel', 'dark', 'colorblind'

print('\n💡 ML Tip: Use "colorblind" palette for accessible publications.')

In [ ]:
# Creating publication-quality figures
fig, ax = plt.subplots(figsize=(10, 6), dpi=100)

# Plot with customization
x = np.linspace(0, 10, 100)
ax.plot(x, np.sin(x), 'b-', linewidth=2, label='sin(x)')
ax.plot(x, np.cos(x), 'r--', linewidth=2, label='cos(x)')

# Professional formatting
ax.set_xlabel('X-axis Label', fontsize=14, fontweight='bold')
ax.set_ylabel('Y-axis Label', fontsize=14, fontweight='bold')
ax.set_title('Publication-Quality Figure', fontsize=16, fontweight='bold', pad=15)

# Legend with shadow and frame
ax.legend(loc='upper right', fontsize=12, shadow=True, fancybox=True)

# Grid
ax.grid(True, alpha=0.3, linestyle='--')

# Tick parameters
ax.tick_params(axis='both', which='major', labelsize=12)

# Spine visibility (optional - remove top/right spines)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Saving figures in different formats
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(df['age'], df['income'], alpha=0.5)
ax.set_xlabel('Age')
ax.set_ylabel('Income')
ax.set_title('Age vs Income')

# Save in multiple formats
# PNG - good for web/presentations
# fig.savefig('plot.png', dpi=300, bbox_inches='tight')

# PDF - good for publications (vector format)
# fig.savefig('plot.pdf', bbox_inches='tight')

# SVG - good for web (vector, editable)
# fig.savefig('plot.svg', bbox_inches='tight')

print('💡 Figure saving tips:')
print('   - Use dpi=300 for print quality')
print('   - Use bbox_inches="tight" to avoid cut-off labels')
print('   - PNG for web/slides, PDF/SVG for publications')
print('   - transparent=True for transparent background')

plt.show()

In [ ]:
# Multi-panel figures with consistent styling
fig = plt.figure(figsize=(14, 10))

# Create grid of subplots with different sizes
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

# Large plot spanning 2 columns
ax1 = fig.add_subplot(gs[0, :2])
sns.histplot(df['income'], kde=True, ax=ax1)
ax1.set_title('A) Income Distribution', fontweight='bold')

# Small plot
ax2 = fig.add_subplot(gs[0, 2])
df['department'].value_counts().plot.pie(autopct='%1.0f%%', ax=ax2)
ax2.set_title('B) Department Split', fontweight='bold')
ax2.set_ylabel('')

# Bottom row - 3 equal plots
ax3 = fig.add_subplot(gs[1, 0])
sns.boxplot(y=df['satisfaction'], ax=ax3)
ax3.set_title('C) Satisfaction', fontweight='bold')

ax4 = fig.add_subplot(gs[1, 1])
sns.boxplot(y=df['performance'], ax=ax4)
ax4.set_title('D) Performance', fontweight='bold')

ax5 = fig.add_subplot(gs[1, 2])
ax5.scatter(df['satisfaction'], df['performance'], alpha=0.5)
ax5.set_xlabel('Satisfaction')
ax5.set_ylabel('Performance')
ax5.set_title('E) Satisfaction vs Performance', fontweight='bold')

fig.suptitle('Multi-Panel Figure with GridSpec', fontsize=16, fontweight='bold', y=1.02)
plt.show()

print('💡 ML Tip: Use multi-panel figures to tell a complete story in one image.')

---

## 9. Practice Exercises

Apply your visualization skills to explore the sample dataset.

### Exercise 1: EDA Dashboard

Create a 2x2 grid of plots showing:
1. Income distribution (histogram with KDE)
2. Performance by department (box plot)
3. Age vs income scatter (colored by department)
4. Correlation heatmap of numeric features

Add proper titles, labels, and a main title for the figure.

In [ ]:
# Exercise 1: Your solution here
# fig, axes = plt.subplots(2, 2, figsize=(14, 12))
# ...


### Exercise 2: Outlier Analysis

1. Create box plots for all numeric columns
2. Identify which columns have outliers
3. Create a scatter plot highlighting outliers in a different color
4. Calculate the percentage of outliers for each feature

In [ ]:
# Exercise 2: Your solution here
# Use IQR method: Q1 - 1.5*IQR and Q3 + 1.5*IQR


### Exercise 3: Feature Relationship Analysis

1. Create a pair plot of: age, income, satisfaction, performance
2. Identify the two most correlated features
3. Create a detailed joint plot for those features
4. Write a brief interpretation of the relationship

In [ ]:
# Exercise 3: Your solution here


### Exercise 4: Class Balance Analysis

1. Create a binary target: `high_income` (income > median)
2. Visualize the class balance
3. Check if the class balance differs by department
4. Create a stacked bar chart showing the proportion in each department

In [ ]:
# Exercise 4: Your solution here


---

## 10. Summary & Quick Reference

### Plot Selection Guide

| Question | Plot Type | Code |
|----------|-----------|------|
| How is one numeric variable distributed? | Histogram, KDE | `sns.histplot()`, `sns.kdeplot()` |
| Are there outliers? | Box plot | `sns.boxplot()` |
| How do two numeric variables relate? | Scatter | `plt.scatter()`, `sns.scatterplot()` |
| How do categories compare? | Bar chart | `sns.countplot()`, `sns.barplot()` |
| How do categories affect a numeric variable? | Box/Violin by category | `sns.boxplot(x='cat', y='num')` |
| What are the correlations? | Heatmap | `sns.heatmap(df.corr())` |
| How do all features relate? | Pair plot | `sns.pairplot()` |

### ML Visualization Checklist

**EDA Phase:**
- [ ] Check target variable distribution
- [ ] Identify missing values with heatmap
- [ ] Plot numeric distributions (look for skewness)
- [ ] Check for outliers with box plots
- [ ] Examine correlations with heatmap
- [ ] Explore categorical variables with count plots

**Model Evaluation Phase:**
- [ ] Plot confusion matrix (classification)
- [ ] Plot learning curves (check overfitting)
- [ ] Plot feature importance
- [ ] Plot residuals (regression)

### Key Takeaways

1. **Always start with simple plots** - histograms and box plots reveal a lot
2. **Check for class imbalance** before training classifiers
3. **Correlation ≠ causation** - but helps with feature selection
4. **Use appropriate scales** - log scale for skewed data
5. **Label everything** - titles, axes, legends for reproducibility

In [ ]:
# Quick reference: Copy-paste templates

print('📋 Quick Copy-Paste Templates:')
print()
print('# Distribution Plot')
print('sns.histplot(df["column"], kde=True)')
print()
print('# Box Plot by Category')
print('sns.boxplot(x="category", y="numeric", data=df)')
print()
print('# Correlation Heatmap')
print('sns.heatmap(df.corr(), annot=True, cmap="RdBu_r", center=0)')
print()
print('# Pair Plot')
print('sns.pairplot(df, hue="category")')
print()
print('# Multi-panel Figure')
print('fig, axes = plt.subplots(2, 2, figsize=(12, 10))')
print('...')
print('plt.tight_layout()')
print()
print('# Save Figure')
print('fig.savefig("plot.png", dpi=300, bbox_inches="tight")')

---

## Next Steps

You've completed the data visualization module! You now know how to:

✅ Create distribution plots (histogram, KDE, box plot)  
✅ Visualize relationships (scatter, pair plot, heatmap)  
✅ Analyze categorical data (bar, count, grouped plots)  
✅ Create ML-specific visualizations (confusion matrix, learning curves)  
✅ Customize and export publication-quality figures  

**Continue to the next notebook:**
- `04_statistics_for_ml.ipynb` - Statistical foundations for machine learning